In [4]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import math
import os
import os
from ultralytics import YOLO
import easyocr
import sys
import os

### Modelo ya entrenado

In [5]:
model = YOLO("system/data/models/best.pt")
reader = easyocr.Reader(['en'])

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


### Video

In [6]:
results_video = model("system/data/videos/Video_10.mkv", save=True)


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/462) C:\Users\andre\Desktop\ALL\Master\AIVA_2026-MUVA\system\data\videos\Video_10.mkv: 384x640 1 ['plate'], 38.5ms
video 1/1 (frame 2/462) C:\Users\andre\Desktop\ALL\Master\AIVA_2026-MUVA\system\data\videos\Video_10.mkv: 384x640 1 ['plate'], 27.5ms
video 1/1 (frame 3/462) C:\Users\andre\Desktop\ALL\Master\AIVA_2026-MUVA\system\data\videos\Video_10.mkv: 384x640 2 ['plate']s, 26.2ms
video 1/1 (frame 4/462) C:\Users\andre\Desktop\ALL\Master\

KeyboardInterrupt: 

### Video frame a frame

In [7]:
cap = cv2.VideoCapture("system/data/videos/Video_10.mkv")
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
frame_count = 0
padding_box = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    # Solo procesar cada 4 frames
    if frame_count % 4 == 0:
        print("frame count: ", frame_count)
        results = model(frame)
        result_frame = results[0]
        # Bucle para recortar
        for box, conf in zip(result_frame.boxes.xyxy, result_frame.boxes.conf):
            if conf < 0.50:
                continue
            # Recortar el box
            x1, y1, x2, y2 = map(int, box)
            x1 = max(0, x1 - padding_box)
            y1 = max(0, y1 - padding_box)
            x2 = min(frame.shape[1], x2 + padding_box)
            y2 = min(frame.shape[0], y2 + padding_box)
            plate_crop = frame[y1:y2, x1:x2]
            # Preprocesado para OCR
            gray = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2GRAY)
            gray = cv2.resize(gray, None, fx=5, fy=5, interpolation=cv2.INTER_CUBIC)
            #gray = cv2.bilateralFilter(gray, 11, 17, 17)
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
            gray = clahe.apply(gray)
            _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            # Aplicar OCR
            result_text = reader.readtext(thresh, detail=1)
            print("@@@@@ Matricula @@@@@@")
            texts = [f"{text}({conf:.2f})" for bbox, text, conf in result_text][::-1]
            print(" ".join(texts))


    if frame_count == 60:
        break

    frame_count += 1

cap.release()
cv2.destroyAllWindows()

frame count:  0

0: 384x640 1 ['plate'], 32.2ms
Speed: 3.2ms preprocess, 32.2ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)
frame count:  4

0: 384x640 1 ['plate'], 26.0ms
Speed: 1.8ms preprocess, 26.0ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)
@@@@@ Matricula @@@@@@
14538(0.64) GLN(0.87)
frame count:  8

0: 384x640 1 ['plate'], 26.4ms
Speed: 1.8ms preprocess, 26.4ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)


C:\Users\andre\Desktop\ALL\Master\AIVA_2026-MUVA\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


@@@@@ Matricula @@@@@@
4638(1.00) GLN(0.71)
frame count:  12

0: 384x640 1 ['plate'], 25.0ms
Speed: 1.3ms preprocess, 25.0ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)
frame count:  16

0: 384x640 1 ['plate'], 25.6ms
Speed: 1.1ms preprocess, 25.6ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)
frame count:  20

0: 384x640 1 ['plate'], 24.6ms
Speed: 1.2ms preprocess, 24.6ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)
@@@@@ Matricula @@@@@@
(L638(0.38) GLN(0.98)
frame count:  24

0: 384x640 1 ['plate'], 26.2ms
Speed: 1.6ms preprocess, 26.2ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)
frame count:  28

0: 384x640 (no detections), 24.4ms
Speed: 1.6ms preprocess, 24.4ms inference, 0.2ms postprocess per image at shape (1, 3, 384, 640)
frame count:  32

0: 384x640 (no detections), 23.9ms
Speed: 1.1ms preprocess, 23.9ms inference, 0.2ms postprocess per image at shape (1, 3, 384, 640)
frame count:  36

0: 384x64